In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sb
from collections import Counter

import warnings
warnings.filterwarnings('ignore')

## Cambiar el formato de los números
pd.set_option('display.float_format', '{:,.0f}'.format)
pd.set_option('display.max_colwidth', None) # configurar vista de las columnas categóricas
pd.set_option('display.max_columns', None) # para no limitar la visualización de la cantidad de columnas

In [2]:
# se debe agregar la ruta apropiada
ruta = '/content/drive/MyDrive/Documentos/UCOM/Diplomado en ciencia de datos - 2023/Proyecto Final/DNCP/data_dncp.csv'
data = pd.read_csv(ruta)
data.shape

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Documentos/UCOM/Diplomado en ciencia de datos - 2023/Proyecto Final/DNCP/data_dncp.csv'

In [ ]:
data.info()

In [ ]:
# Convertir las variables de fechas en tipo datetime
data['compiledRelease/contracts/0/period/endDate'] = pd.to_datetime(data['compiledRelease/contracts/0/period/endDate'])
data['compiledRelease/contracts/0/period/startDate'] = pd.to_datetime(data['compiledRelease/contracts/0/period/startDate'])
data['compiledRelease/contracts/0/dateSigned'] = pd.to_datetime(data['compiledRelease/contracts/0/dateSigned'])

# Calcular el periodo de los contratos en días, y luego llenar los campos nulos de la variable "compiledRelease/contracts/0/period/durationInDays" con
# estos valores
data['cant_dias_aux'] = (data['compiledRelease/contracts/0/period/endDate']-data['compiledRelease/contracts/0/period/startDate']).dt.days
data['compiledRelease/contracts/0/period/durationInDays'].fillna(data['cant_dias_aux'], inplace=True)

# Eliminar los valores nulos de la columna "compiledRelease/contracts/0/period/durationInDays"
data = data[data['compiledRelease/contracts/0/period/durationInDays'].isnull()==False]
data = data[data['compiledRelease/tender/numberOfTenderers'].isnull()==False]


# Sustituir  las categorías de la variable "compiledRelease/tender/mainProcurementCategoryDetails" por valores menos extensos.
categorias = data['compiledRelease/tender/mainProcurementCategoryDetails'].unique().tolist()
new_categorias = [f'cat_{i+1}' for i in range(len(categorias))]
mapping = {}
for i in range(len(categorias)):
  mapping[categorias[i]]=new_categorias[i]
data['compiledRelease/tender/mainProcurementCategoryDetails'] = data['compiledRelease/tender/mainProcurementCategoryDetails'].replace(mapping)


## Eliminar las categorías con frecuencia < 30.
categorias_para_analisis = []
for cat in new_categorias:
  freq = data[data['compiledRelease/tender/mainProcurementCategoryDetails']==cat].shape[0]
  if freq > 30:
    categorias_para_analisis.append(cat)

data = data[data['compiledRelease/tender/mainProcurementCategoryDetails'].isin(categorias_para_analisis)].reset_index(drop=True)

## Sustituir las categorías de la variable "'compiledRelease/tender/procurementMethodDetails" por unos alias.
categorias = data['compiledRelease/tender/procurementMethodDetails'].unique().tolist()
mapping = {'Concurso de Ofertas':'CO', 'Contratación Directa':'CD', 'Licitación Pública Nacional':'LPN'}
data['compiledRelease/tender/procurementMethodDetails'] = data['compiledRelease/tender/procurementMethodDetails'].replace(mapping)

## Generar variables asociadas a la duración de los contratos
lista_duracion_dias = data['compiledRelease/contracts/0/period/durationInDays'].tolist()
duracion_contrato_semanas = [round(x/7, 1) for x in lista_duracion_dias ]
duracion_contrato_meses = [round(x/30, 1) for x in lista_duracion_dias]

data['duracion_contrato_semanas'] = duracion_contrato_semanas
data['duracion_contrato_meses'] = duracion_contrato_meses

## Selecionar solamente las columnas relevantes para el análisis y el modelo.
columnas_para_el_modelo =  ['compiledRelease/contracts/0/id',
                            'compiledRelease/tender/mainProcurementCategoryDetails',
                            'compiledRelease/tender/tenderPeriod/durationInDays',
                            'compiledRelease/tender/procurementMethodDetails',
                            'compiledRelease/tender/numberOfTenderers',
                            'compiledRelease/contracts/0/period/durationInDays',
                            'compiledRelease/contracts/0/value/amount',
                            'duracion_contrato_semanas',
                            'duracion_contrato_meses'
                            ]
data = data[columnas_para_el_modelo]

## Asignar nombres más apropiados a las columnas.
data.rename(columns = {
    'compiledRelease/contracts/0/id':'id_contrato',
    'compiledRelease/tender/mainProcurementCategoryDetails':'categoria_llamado',
    'compiledRelease/tender/tenderPeriod/durationInDays': 'duracion_llamado_dias',
    'compiledRelease/tender/procurementMethodDetails': 'tipo_licitacion',
    'compiledRelease/tender/numberOfTenderers' : 'nro_oferentes',
    'compiledRelease/contracts/0/period/durationInDays': 'duracion_contrato_dias',
    'compiledRelease/contracts/0/value/amount': 'monto_contrato'}, inplace = True
            )

In [ ]:
data.shape

In [ ]:
data.sample(7)

In [ ]:
## Generar dataframes para cada tipo de Licitación
data_lpn = data[data['tipo_licitacion']=='LPN'].reset_index(drop=True)
data_co = data[data['tipo_licitacion']=='CO'].reset_index(drop=True)
data_cd = data[data['tipo_licitacion']=='CD'].reset_index(drop=True)

In [ ]:
print('data_lpn.shape: ', data_lpn.shape)
print('data_co.shape:', data_co.shape)
print('data_cd.shape:', data_cd.shape)


In [ ]:
data_lpn.sample(2)

In [ ]:
data_lpn.to_csv('data_lpn.csv', index=False)
data_co.to_csv('data_co.csv', index=False)
data_cd.to_csv('data_cd.csv', index=False)